In [2]:
# Add the path to the src module


from extract_pLDDT_functions import extract_protein_seq_pdb_gz
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO

import sys
from pathlib import Path
import shutil

sys.path.append(
    '/data/c/yangyusheng_optimized/AFdb_based_domain_extraction/src/util')
# load basic python modules

# 1.Load AF data

In [3]:
AF_pdb_dir = Path(
    "/data/c/yangyusheng_optimized/resource/AlphaFold_Dataset/20230708_downloaded/UP000002485_284812_SCHPO_v4")

AF_protein_stats = pd.DataFrame()

for pdb_gz_file in AF_pdb_dir.glob("*.pdb.gz"):

    uniprot_id = pdb_gz_file.stem.split("-")[1]
    protein_seq = extract_protein_seq_pdb_gz(pdb_gz_file)

    AF_protein_stats.loc[uniprot_id, "AF_seq"] = str(protein_seq)

AF_protein_stats["AF_length"] = AF_protein_stats["AF_seq"].apply(len)

AF_protein_stats_df = AF_protein_stats.rename_axis("uniprot_id").reset_index()

# 2.Load pombase data

In [4]:
peptide_file = Path(
    "/data/c/yangyusheng_optimized/resource/Pombase_FTP/release/20230701/peptide.fa")

# Parse the FASTA file and convert to DataFrame
records = list(SeqIO.parse(peptide_file, "fasta"))
data = [(re.search("(\S+)\.\d{1}:pep", record.id).group(1),
         str(record.seq)) for record in records]
pombase_protein_stats = pd.DataFrame(
    data, columns=["gene_systematic_id", "Pombase_seq"])
pombase_protein_stats["Pombase_seq"] = pombase_protein_stats["Pombase_seq"].str.rstrip(
    "*")
pombase_protein_stats["Pombase_length"] = pombase_protein_stats["Pombase_seq"].apply(
    len)

# 3.Load the connection information

In [5]:
gene_info = pd.read_csv(
    "/data/c/yangyusheng_optimized/resource/Pombase_FTP/release/20230701/gene_IDs_names_products.tsv", sep="\t", index_col=0)
gene_info["gene_name"] = gene_info["gene_name"].fillna(
    gene_info.index.to_series()
)
gene_info.reset_index(drop=False, inplace=True)

coding_genes_info = gene_info.query(
    "gene_type == 'protein coding gene'").copy()

# 4.Merge data

In [6]:
coding_genes = coding_genes_info.merge(pombase_protein_stats, on="gene_systematic_id", how="outer").merge(
    AF_protein_stats_df, on="uniprot_id", how="outer")

In [7]:
# merged_data[merged_data[["Pombase_seq", "AF_seq"]].notna().any(axis=1)].to_csv(
#     "../results/compare_AF_pombase.csv", index=False, header=True)

# 5.Compare

In [8]:
def assign_type(row):

    Pombase_seq = row["Pombase_seq"]
    AF_seq = row["AF_seq"]
    Pombase_length = row["Pombase_length"]
    AF_length = row["AF_length"]

    if pd.notna(Pombase_seq) and pd.isna(AF_seq):
        return "no_prediction_proteins"
    elif pd.isna(Pombase_seq) and pd.notna(AF_seq):
        return "no_Pombase_proteins"
    elif pd.isna(Pombase_seq) and pd.isna(AF_seq):
        return "no_proteins"
    elif Pombase_seq == AF_seq:
        return "same_seq"
    elif (Pombase_seq != AF_seq) and (Pombase_length == AF_length):
        return "different_seq_same_length"
    elif (Pombase_seq != AF_seq) and (Pombase_length != AF_length):
        return "different_seq_different_length"
    else:
        return "undefined"


coding_genes["type"] = coding_genes.apply(assign_type, axis=1)
# coding_genes.to_excel(
#     "../results/coding_genes_AF_vs_pombase_summary.xlsx", index=False, header=True)
print(coding_genes["type"].value_counts())

type
same_seq                          4954
different_seq_different_length     134
no_prediction_proteins              28
no_Pombase_proteins                 26
different_seq_same_length           18
Name: count, dtype: int64


# 6.Output the fasta file for the inconsistent proteins

In [9]:
def write_fasta(df, outputFolder):
    outputFolder.mkdir(parents=True, exist_ok=True)
    for index, row in df.iterrows():
        with open(outputFolder / (row["gene_systematic_id"] + ".fasta"), "w") as f:
            f.write(">" + row["gene_systematic_id"] + "\n")
            f.write(row["Pombase_seq"] + "\n")

In [10]:
fasta_output_folder = Path("../results/Fasta_for_customed_prediction")
fasta_output_folder.mkdir(exist_ok=True)

aa_change_proteins = coding_genes.query("type == 'different_seq_same_length'")
different_seq_proteins = coding_genes.query(
    "type == 'different_seq_different_length'")
no_prediction_proteins = coding_genes.query("type == 'no_prediction_proteins'")

selected_proteins_for_customed_prediction = [
    "SPAC4F8.12c", "SPAC3H5.09c", "SPBC1289.08", "SPAC890.02c", "SPBC21C3.16c", "SPAC3F10.19"]
selected_proteins_for_customed_prediction_df = coding_genes[coding_genes["gene_systematic_id"].isin(
    selected_proteins_for_customed_prediction)].copy()

with pd.ExcelWriter("../results/coding_genes_AF_vs_pombase_summary.xlsx") as xlsx:
    coding_genes.to_excel(
        xlsx, sheet_name="coding_genes_AF_vs_pombase", index=False, header=True)
    aa_change_proteins.to_excel(
        xlsx, sheet_name="aa_change_proteins", index=False, header=True)
    different_seq_proteins.to_excel(
        xlsx, sheet_name="different_len_proteins", index=False, header=True)
    no_prediction_proteins.to_excel(
        xlsx, sheet_name="no_prediction_proteins", index=False, header=True)
    selected_proteins_for_customed_prediction_df.to_excel(
        xlsx, sheet_name="selected_proteins", index=False, header=True)

write_fasta(selected_proteins_for_customed_prediction_df, fasta_output_folder /
            "selected_proteins_for_customed_prediction")
write_fasta(aa_change_proteins, fasta_output_folder /
            "aa_change_proteins")
write_fasta(different_seq_proteins, fasta_output_folder /
            "different_length_and_different_seq_proteins")
write_fasta(no_prediction_proteins, fasta_output_folder /
            "no_prediction_proteins")

# 7.re-predict alphafold results

In [11]:
predicted_results_folder = Path(
    "../data/0_raw/20240724_predicted_results_renamed")

AFdb_vDulab_results_merged_folder = Path(
    "../data/1_AFdb_vDulab_results_merged")
AFdb_vDulab_results_merged_folder.mkdir(exist_ok=True)
(AFdb_vDulab_results_merged_folder / "PDB").mkdir(exist_ok=True)
(AFdb_vDulab_results_merged_folder / "PAE_files").mkdir(exist_ok=True)


AFdb_PDB_folder = AF_pdb_dir
AFdb_PAE_folder = AF_pdb_dir.parent / "PAE_files"
vDulab_PDB_folder = predicted_results_folder / "vDulab"
vDulab_PAE_folder = predicted_results_folder / "PAE_files"

AFdb_PDB_files = {
    file.name.split("-")[1]: file for file in AFdb_PDB_folder.glob("*.pdb.gz")
}
AFdb_PAE_files = {
    file.name.split("-")[1]: file for file in AFdb_PAE_folder.glob("*.json")
}
vDulab_PDB_files = {
    file.name.split("-")[1]: file for file in vDulab_PDB_folder.glob("*.pdb.gz")
}
vDulab_PAE_files = {
    file.name.split("-")[1]: file for file in vDulab_PAE_folder.glob("*.json")
}

for idx, row in coding_genes.iterrows():
    uniprot_id = row["uniprot_id"]
    gene_systematic_id = row["gene_systematic_id"]
    seq_type = row["type"]

    if seq_type == "same_seq":
        pdb_file = AFdb_PDB_files.get(uniprot_id)
        pae_file = AFdb_PAE_files.get(uniprot_id)
        coding_genes.loc[idx, "Source"] = "AFdb"
    elif seq_type == "different_seq_same_length":
        pdb_file = vDulab_PDB_files.get(uniprot_id)
        pae_file = vDulab_PAE_files.get(uniprot_id)
        coding_genes.loc[idx, "Source"] = "vDulab"
    elif seq_type == "no_prediction_proteins":
        pdb_file = vDulab_PDB_files.get(uniprot_id)
        pae_file = vDulab_PAE_files.get(uniprot_id)
        coding_genes.loc[idx, "Source"] = "vDulab"
    elif seq_type == "different_seq_different_length":
        pdb_file = vDulab_PDB_files.get(uniprot_id)
        pae_file = vDulab_PAE_files.get(uniprot_id)
        coding_genes.loc[idx, "Source"] = "vDulab"

    # if pdb_file:
    #     shutil.copy(pdb_file, AFdb_vDulab_results_merged_folder / "PDB")
    # if pae_file:
    #     shutil.copy(pae_file, AFdb_vDulab_results_merged_folder / "PAE_files")

    # pdb_file.copy(AFdb_vDulab_results_merged_folder / "PDB")
    # pae_file.copy(AFdb_vDulab_results_merged_folder / "PAE_files")

In [12]:
aa_change_proteins = coding_genes.query("type == 'different_seq_same_length'")
different_seq_proteins = coding_genes.query(
    "type == 'different_seq_different_length'")
no_prediction_proteins = coding_genes.query("type == 'no_prediction_proteins'")

selected_proteins_for_customed_prediction = [
    "SPAC4F8.12c", "SPAC3H5.09c", "SPBC1289.08", "SPAC890.02c", "SPBC21C3.16c", "SPAC3F10.19"]
selected_proteins_for_customed_prediction_df = coding_genes[coding_genes["gene_systematic_id"].isin(
    selected_proteins_for_customed_prediction)].copy()

with pd.ExcelWriter("../results/coding_genes_AF_vs_pombase_summary.xlsx") as xlsx:
    coding_genes.to_excel(
        xlsx, sheet_name="coding_genes_AF_vs_pombase", index=False, header=True)
    aa_change_proteins.to_excel(
        xlsx, sheet_name="aa_change_proteins", index=False, header=True)
    different_seq_proteins.to_excel(
        xlsx, sheet_name="different_len_proteins", index=False, header=True)
    no_prediction_proteins.to_excel(
        xlsx, sheet_name="no_prediction_proteins", index=False, header=True)
    selected_proteins_for_customed_prediction_df.to_excel(
        xlsx, sheet_name="selected_proteins", index=False, header=True)

In [13]:

# # pLDDT_score = pd.read_csv(
# #     "../results/UP000002485_284812_SCHPO_v4_pLDDT_scores.tsv", sep="\t")
# # pLDDT_score["mean_pLDDT"] = pLDDT_score["pLDDT"].str.split(
# #     ",", expand=True).astype(float).mean(axis=1).round(2)


# def count_differences(str1, str2):
#     return sum(c1 != c2 for c1, c2 in zip(str1, str2))


# coding_genes.loc[(coding_genes["same_length_different_seq"] == "YES"), "same_length_different_seq_aa_count"] = coding_genes.loc[(
#     coding_genes["same_length_different_seq"] == "YES"), :].apply(lambda row: count_differences(row["Pombase_seq"], row["AF_seq"]), axis=1)

# coding_genes["length_difference"] = coding_genes["Pombase_length"] - \
#     coding_genes["AF_length"]

# coding_genes["Pombase_seq_start_codon"] = coding_genes["Pombase_seq"].str[0]
# coding_genes["AF_seq_start_codon"] = coding_genes["AF_seq"].str[0]

# coding_genes = coding_genes.merge(
#     pLDDT_score[["uniprot_id", "mean_pLDDT"]], on="uniprot_id", how="left")